In [ ]:
import geopandas as gpd
from pathlib import Path
import fiona
import matplotlib.pyplot as plt

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
networks_folder = base_path / "Processed_data/networks"

# Define the output directory path
networks_catchments_intersections = base_path / "Processed_data/networks/networks_catchments_intersections"


In [ ]:
jamaica_metric_grid_crs = "EPSG:3448"

In [ ]:
jamaica_boundary_path = base_path / "Inputs/Boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(f"Original Jamaica boundary CRS: {jamaica_boundary.crs}")

In [ ]:
# Path to your ports GeoPackage
ports_gpkg_path = networks_folder / "transport/port_polygon.gpkg"
layers = fiona.listlayers(ports_gpkg_path)
print("Available layers:", layers)

In [ ]:
hydrobasins = base_path / "Processed_data/HydroBASINS_Level12_Clipped_Jamaica.shp"
hydrobasins = gpd.read_file(hydrobasins)
print(hydrobasins.crs)

In [ ]:
# Load the rail data (edges first)
ports_areas = gpd.read_file(ports_gpkg_path)
ports_areas = ports_areas.to_crs(jamaica_metric_grid_crs)

In [ ]:
ports_areas.columns

In [ ]:
# === For Point Features (Road Nodes) ===
# Perform a spatial join to attach hydrobasin attributes (e.g., HYBAS_ID) to each node.
ports_areas_join = gpd.sjoin(ports_areas, hydrobasins, how="left", predicate="intersects")

display(ports_areas_join.head())
display(ports_areas_join.columns)

# Example aggregation: Count the number of nodes per catchment
nodes_count_by_catchment = ports_areas_join.groupby("HYBAS_ID").size().reset_index(name="node_count")
display("\nNode Count by Catchment:")
display(nodes_count_by_catchment)

In [ ]:
# Save the nodes join layer to a new GeoPackage
ports_areas_catchments_intersection = networks_catchments_intersections / "ports_areas_catchments_intersection.gpkg"
ports_areas_join.to_file(ports_areas_catchments_intersection, layer="joined_nodes", driver="GPKG")